# Contracts variables

This notebook has the purpose to:

-use proxies for missing variables
-create needed variables for analysis

In [1]:
import pandas as pd
import numpy as np
from pyprojroot import here
import benford as bf
import matplotlib.pyplot as plt
import seaborn as sns
import utils


figures_folder = here('figures/')
base_name_figures = '3addvariables_'
processed_data = here('data/processed_data')

# Loading data

In [2]:
var2keep = [
    #identifiers
    'contract_year',
    'file_code',
    'contract_code',
    'purchasing_unit_id',
    'supplier_name_clean',
    #variables
    'submission_deadline_date',
    'advertisement_date',
    'award_date',
    'contract_signature_date',
    'contract_end_date', 
    'contract_initial_date', 
    'contract_price_mx',
    'government_level', 
    'legal_framework',
    'legal_fundament_simplified', 
    'procedure_type_fixed',
    'procedure_venue', 
    'supplier_size',
    'supply_type']

#read feather
contracts = pd.read_feather(processed_data / 'mxc11to22_base.feather', columns=var2keep)
contracts.shape


(2308908, 19)

In [3]:
contracts['prov_id'] = contracts.index

In [4]:
(contracts.isnull().sum() / contracts.shape[0]).sort_values(ascending=False)

legal_fundament_simplified    0.644734
award_date                    0.595893
contract_signature_date       0.591509
supplier_size                 0.290171
submission_deadline_date      0.282378
advertisement_date            0.173678
procedure_venue               0.029436
legal_framework               0.029391
supply_type                   0.003249
procedure_type_fixed          0.002651
contract_end_date             0.000141
contract_year                 0.000000
contract_initial_date         0.000000
government_level              0.000000
contract_price_mx             0.000000
file_code                     0.000000
supplier_name_clean           0.000000
purchasing_unit_id            0.000000
contract_code                 0.000000
prov_id                       0.000000
dtype: float64

# Benford law / 'mad_per_year'

In [5]:
contracts

,contract_year,file_code,contract_code,purchasing_unit_id,supplier_name_clean,submission_deadline_date,advertisement_date,award_date,contract_signature_date,contract_end_date,contract_initial_date,contract_price_mx,government_level,legal_framework,legal_fundament_simplified,procedure_type_fixed,procedure_venue,supplier_size,supply_type,prov_id
0,2011,118647,886089,050GYR005,tcaempresarialsadecv,2011-12-15,NaT,NaT,NaT,2011-12-31,2011-12-14,146343.00,federal_authority,international,None,direct_failed_open,mixed,small,goods,0
1,2011,118978,886108,050GYR005,instrumentosyaccesoriosautomatizadossadecv,2011-12-16,NaT,NaT,NaT,2011-12-25,2011-12-15,6250.00,federal_authority,international,None,direct_failed_open,mixed,small,goods,1
2,2011,121476,836635,050GYR025,dacegacorporation,2011-12-16,NaT,NaT,2011-12-22,2011-12-29,2011-12-22,54400.00,federal_authority,international,None,direct_failed_open,in-person,small,goods,2
3,2011,122542,74897,016000997,lizbethapariciorazo,2011-09-19,NaT,NaT,2011-09-29,2011-10-12,2011-10-03,88579.20,federal_authority,international,None,direct_failed_open,mixed,small,goods,3
4,2011,124569,76721,821045953,victormiguelvallejojuarez,2011-09-13,NaT,NaT,2011-09-19,2011-12-07,2011-09-20,163970.68,local_authority,national,None,at_least_three,in-person,None,services,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2308903,2022,2022-15-QDV-00000007,2022-15-QDV-000000072022-02-122022-02-19,QDV,formaseficientessadecv,NaT,NaT,NaT,NaT,2022-02-19,2022-02-12,364613.67,federal_authority,national,art41,direct,electronic,None,goods,2308903
2308904,2022,2022-15-QDV-00000008,2022-15-QDV-000000082022-02-142022-02-21,QDV,cosmopapelsadecv,NaT,NaT,NaT,NaT,2022-02-21,2022-02-14,346016.40,federal_authority,national,art41,direct,electronic,None,goods,2308904
2308905,2022,2022-15-QDV-00000009,2022-15-QDV-000000092022-02-112022-02-18,QDV,farvisaninsumosinstitucionalessadecv,NaT,NaT,NaT,NaT,2022-02-18,2022-02-11,1885.35,federal_authority,national,art41,direct,electronic,None,goods,2308905
2308906,2022,2022-12-NBD-00000001,2022-12-NBD-000000012022-02-222022-02-27,NBD,formaseficientessadecv,NaT,NaT,NaT,NaT,2022-02-27,2022-02-22,1036815.04,federal_authority,national,art41,direct,electronic,None,goods,2308906


In [6]:
#open json file mad_peryear.json
import json
with open(processed_data / 'mad_peryear.json') as json_file:
    mad_peryear = json.load(json_file)


In [7]:
#import data from feather
mad_df = pd.DataFrame(mad_peryear)
mad_df.head()
mad_df['mad'] = mad_df['mad'].astype(float)
mad_df['contract_year'] = mad_df['contract_year'].astype(int)
mad_df['purchasing_unit_id'] = mad_df['purchasing_unit_id'].astype(str)


In [8]:
mad_df

,contract_year,purchasing_unit_id,mad
0,2011,002000999,0.019427
1,2011,004000995,0.020253
2,2011,004000997,0.022153
3,2011,004000998,0.038268
4,2011,004D00001,0.033540
...,...,...,...
7151,2022,GYN,0.022680
7152,2022,GYR,0.029321
7153,2022,P7R,0.023036
7154,2022,PBE,0.013470


In [9]:
contracts = contracts.merge(mad_df, on=['contract_year', 'purchasing_unit_id'], how='left')

# CRI

In [10]:
var2keep = [
    #identifiers
    'prov_id',
    #'supplier_name_clean',
    #'contract_initial_date',
    #'contract_price_mx',
    #'purchasing_unit_id',
    #'contract_year',

    #red flags
    #'CRI',
    'rf_submission_period',
    'rf_decision_period',
    'rf_procedure_type',
    'rf_bl_conformity',
    'rf_buyer_dependence',
    'rf_single_bidder',

    #red flags components
    #'submission_period_t',
    #'decision_period_t'
    ]

In [11]:
#read
cri_df = pd.read_feather(processed_data / 'mxc_11to22_cri.feather', columns=var2keep)

In [12]:
contracts = contracts.merge(cri_df, on = ['prov_id'], how = 'left')

# Czibik_etal_2021

In [13]:
#read
czibik_df = pd.read_feather(processed_data / 'czibik_etal2021.feather')

In [14]:
czibik_df = czibik_df.drop_duplicates(subset=['contract_year', 'purchasing_unit_id', 'supplier_name_clean'])

In [15]:
czibik_df.columns

Index(['supplier_name_clean', 'purchasing_unit_id', 'contract_year',
       'buyer_cri', 'supplier_cri', 'bs_cri', 'neighbourhood_cri'],
      dtype='object')

In [16]:
print(contracts.shape)
contracts = contracts.merge(czibik_df, on = ['contract_year', 'purchasing_unit_id', 'supplier_name_clean'], how = 'left')
print(contracts.shape)

(2308908, 27)
(2308908, 31)


# Fazekas_wachs_2020

In [17]:
#import feathers file
fazekaswachs2020 = pd.read_feather(processed_data / 'fazekaswachs2020.feather')

In [18]:
fazekaswachs2020.shape

(28315, 9)

In [19]:
fazekaswachs2020[['contract_year', 'purchasing_unit_id']].drop_duplicates().shape

(28315, 2)

In [20]:
fazekaswachs2020.columns

Index(['contract_year', 'purchasing_unit_id', 'entropy_buyer_dependence', 'p3',
       'c4', 'unweighted_competitive_clustering', 'allcycles_summed_geomeans',
       'unbounded_weighted_competitive_clustering',
       'bounded_weighted_competitive_clustering'],
      dtype='object')

In [21]:
fazekaswachs2020.isnull().sum()

contract_year                                    0
purchasing_unit_id                               0
entropy_buyer_dependence                         0
p3                                               0
c4                                               0
unweighted_competitive_clustering             3507
allcycles_summed_geomeans                    11649
unbounded_weighted_competitive_clustering    11649
bounded_weighted_competitive_clustering      11649
dtype: int64

In [22]:
fazekaswachs2020['bounded_weighted_competitive_clustering'] = fazekaswachs2020['bounded_weighted_competitive_clustering'].fillna(0)
fazekaswachs2020['unbounded_weighted_competitive_clustering'] = fazekaswachs2020['unbounded_weighted_competitive_clustering'].fillna(0)
fazekaswachs2020['unweighted_competitive_clustering'] = fazekaswachs2020['unweighted_competitive_clustering'].fillna(0)


In [23]:
print(contracts.shape)
contracts = contracts.merge(fazekaswachs2020[['purchasing_unit_id', 'contract_year', 'entropy_buyer_dependence', 'unweighted_competitive_clustering' , 'unbounded_weighted_competitive_clustering', 'bounded_weighted_competitive_clustering' ]], how='left', on=['contract_year', 'purchasing_unit_id'])
print(contracts.shape)

(2308908, 31)
(2308908, 35)


# IMCO

In [24]:
#import feathers file
imco_df = pd.read_feather(processed_data / 'imco.feather')

In [25]:
imco_df.columns

Index(['supplier_name_clean', 'purchasing_unit_id', 'contract_year', 'prov_id',
       'prop_b_direct', 'prop_b_failedopen', 'prop_b_direct_full',
       'prop_s_direct', 'prop_s_failedopen', 'prop_s_direct_full', 'notinRUPC',
       'ncontracts_bsw', 'active_weeks_bs', 'spending_bs_aw', 'ncontracts_bs',
       'ncontracts_bs_odds', 'fragmented_contract_odds', 'prop_contracts_bs'],
      dtype='object')

In [26]:
imco_df.shape

(2308908, 18)

In [27]:
print(contracts.shape)
contracts = contracts.drop(columns=['supplier_name_clean', 'purchasing_unit_id', 'contract_year']).merge(imco_df, how='left', on=['prov_id'])
print(contracts.shape)

(2308908, 35)
(2308908, 49)


# Projection centralities

In [28]:
projection_centralities_data = here('data/processed_data/net_projection_data')

In [29]:
cyear = [2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022]
supplier_all = pd.DataFrame()
buyer_all = pd.DataFrame()
for i in cyear:
    supplier_c = pd.read_feather(projection_centralities_data / f'centralities_supplier_{i}.feather')
    buyer_c = pd.read_feather(projection_centralities_data / f'centralities_buyer_{i}.feather')
    #rename norm_betweenness to betweenness_norm
    supplier_c = supplier_c.rename(columns={'norm_betweenness': 'betweenness_norm'})
    buyer_c = buyer_c.rename(columns={'norm_betweenness': 'betweenness_norm'})
    #normalize supplier
    supplier_c['degree_norm'] = supplier_c['degree'] / (supplier_c['name'].nunique() - 1)
    supplier_c['strength_norm'] = supplier_c['strength'] / ((supplier_c['name'].nunique() - 1)*(buyer_c['name'].nunique() - 1))
    #normalize buyer
    buyer_c['degree_norm'] = buyer_c['degree'] / (buyer_c['name'].nunique() - 1)
    buyer_c['strength_norm'] = buyer_c['strength'] / ((supplier_c['name'].nunique() - 1)*(buyer_c['name'].nunique() - 1))
    #concatenate
    cols2drop = ['id', 'betweenness', 'degree', 'strength']
    supplier_c = supplier_c.drop(columns=cols2drop)
    buyer_c = buyer_c.drop(columns=cols2drop)
    supplier_c = supplier_c.rename(columns={'name': 'supplier_name_clean'})
    buyer_c = buyer_c.rename(columns={'name': 'purchasing_unit_id'})
    supplier_all = pd.concat([supplier_all, supplier_c])
    buyer_all = pd.concat([buyer_all, buyer_c])

In [30]:
# Columns you want to add a suffix to
columns_to_rename = ['betweenness_norm', 'closeness', 'eigen', 'degree_norm', 'strength_norm']
suffix_s = '_supplier'
suffix_b = '_buyer'
# Add suffix to the columns in supplier_all DataFrame
supplier_all = supplier_all.rename(columns={col: col + suffix_s for col in columns_to_rename})
# Add suffix to the columns in buyer_all DataFrame
buyer_all = buyer_all.rename(columns={col: col + suffix_b for col in columns_to_rename})

In [31]:
print(contracts.shape)
contracts = contracts.merge(supplier_all, how='left', on=['contract_year', 'supplier_name_clean'])
print(contracts.shape)
contracts = contracts.merge(buyer_all, how='left', on=['contract_year', 'purchasing_unit_id'])
print(contracts.shape)

(2308908, 49)
(2308908, 54)
(2308908, 59)


# Bipartite data

In [32]:
bipartite_data = here('data/processed_data/net_bipartite_data')

In [33]:
def seq_coreness(df, col1, col2):
    all_cores = set(df[col1]).union(set(df[col2])) #check if there are any common values in the two columns
    if len(all_cores) == max(all_cores):
        replacement_dict = {i: i for i in all_cores}
    else:
        replacement_dict = {value: (index + 1) for index, value in enumerate(sorted(all_cores))}

    h_core = max(replacement_dict.values())

    return replacement_dict, h_core

In [34]:
cyear = [2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022]
bipartite_all = pd.DataFrame()
for i in cyear:
    bipartite_c = pd.read_feather(bipartite_data / f'bipartite_{i}.feather')
    #edge betweenness
    total_nodes = bipartite_c['purchasing_unit_id'].nunique() + bipartite_c['supplier_name_clean'].nunique()
    normalization_factor = 2 / (total_nodes * (total_nodes - 1))
    bipartite_c['betweenness_edge_norm'] = bipartite_c['edge_betweenness'] / normalization_factor
    #coreness
    #wdeg
    wdeg_dict, wdeg_hcore = seq_coreness(bipartite_c, 'corewdeg_b', 'corewdeg_s')
    bipartite_c['corewdeg_b'] = bipartite_c['corewdeg_b'].map(wdeg_dict) / wdeg_hcore
    bipartite_c['corewdeg_s'] = bipartite_c['corewdeg_s'].map(wdeg_dict) / wdeg_hcore
    #ad, averagedegree
    ad_dict, ad_hcore = seq_coreness(bipartite_c, 'coread_b', 'coread_s')
    bipartite_c['coread_b'] = bipartite_c['coread_b'].map(ad_dict) / ad_hcore
    bipartite_c['coread_s'] = bipartite_c['coread_s'].map(ad_dict) / ad_hcore
    #strength
    strength_dict, strength_hcore = seq_coreness(bipartite_c, 'corestrength_b', 'corestrength_s')
    bipartite_c['corestrength_b'] = bipartite_c['corestrength_b'].map(strength_dict) / strength_hcore
    bipartite_c['corestrength_s'] = bipartite_c['corestrength_s'].map(strength_dict) / strength_hcore
    #drop
    cols2drop = ['weight', 'buyer_id', 'supplier_id', 'c_id_lg', 'edge_betweenness']
    #concat
    bipartite_c = bipartite_c.drop(columns=cols2drop)
    bipartite_all = pd.concat([bipartite_all, bipartite_c])
        

In [35]:
bipartite_all.shape

(1059184, 10)

In [36]:
#merge
print(contracts.shape)
contracts = contracts.merge(bipartite_all, how='left', on=['contract_year', 'supplier_name_clean', 'purchasing_unit_id'])
print(contracts.shape)

(2308908, 59)
(2308908, 66)


## My variables 

### proportion of direct procedures in neighborhood

In [37]:
#additional variables
#proportion of direct contracts
contracts['procedure_type'] = np.where(contracts['procedure_type_fixed'] == 'direct_failed_open', 'direct', contracts['procedure_type_fixed'])

In [38]:
contracts['procedure_type_direct_dummy'] = np.where(contracts['procedure_type'] == 'direct', 1, 0)

In [39]:
contracts['procedure_type_direct_dummy'].value_counts(normalize=True)

1    0.748071
0    0.251929
Name: procedure_type_direct_dummy, dtype: float64

In [40]:
#get the agggreated proportion of direct contracts
bs_prop_direct = contracts.groupby(['contract_year', 'purchasing_unit_id', 'supplier_name_clean'])['procedure_type_direct_dummy'].mean().reset_index()
#rename
bs_prop_direct = bs_prop_direct.rename(columns={'procedure_type_direct_dummy': 'bs_prop_direct'})
bs_prop_direct.shape

(1059184, 4)

In [41]:
bs_prop_direct_buyer = bs_prop_direct.groupby(['contract_year', 'purchasing_unit_id']).agg(bs_pd_buyer_sum = ('bs_prop_direct', 'sum'), nsuppliers = ('bs_prop_direct', 'count')).reset_index()

In [42]:
bs_prop_direct_supplier = bs_prop_direct.groupby(['contract_year', 'supplier_name_clean']).agg(bs_pd_supplier_sum = ('bs_prop_direct', 'sum'), nbuyers = ('bs_prop_direct', 'count')).reset_index()

In [43]:
print(bs_prop_direct.shape)
print(bs_prop_direct_buyer.shape)
print(bs_prop_direct_supplier.shape)

(1059184, 4)
(28315, 4)
(629582, 4)


In [44]:
bs_prop_direct = bs_prop_direct.merge(bs_prop_direct_buyer, on=['contract_year', 'purchasing_unit_id'], how='left')

In [45]:
bs_prop_direct = bs_prop_direct.merge(bs_prop_direct_supplier, on=['contract_year', 'supplier_name_clean'], how='left')

In [46]:
print(bs_prop_direct.shape)

(1059184, 8)


In [47]:
bs_prop_direct['bs_pd_no_intersection'] = bs_prop_direct['bs_pd_buyer_sum'] + bs_prop_direct['bs_pd_supplier_sum'] - (2*bs_prop_direct['bs_prop_direct'])

In [48]:
bs_prop_direct['n_neighbors'] = bs_prop_direct['nsuppliers'] + bs_prop_direct['nbuyers'] - 2

In [49]:
bs_prop_direct['neighborhood_propdirect'] = bs_prop_direct['bs_pd_no_intersection'] / bs_prop_direct['n_neighbors']

In [50]:
bs_prop_direct['neighborhood_propdirect'].describe()

count    1.057353e+06
mean     6.595532e-01
std      3.115503e-01
min      0.000000e+00
25%      4.801670e-01
50%      7.580404e-01
75%      9.197368e-01
max      1.000000e+00
Name: neighborhood_propdirect, dtype: float64

In [51]:
contracts.drop(columns=['procedure_type', 'procedure_type_direct_dummy'], inplace=True)

In [52]:
contracts = contracts.merge(bs_prop_direct[['contract_year', 'purchasing_unit_id', 'supplier_name_clean', 'bs_prop_direct', 'neighborhood_propdirect']], on=['contract_year', 'purchasing_unit_id', 'supplier_name_clean'], how='left')

### Tender period

In [53]:
print(contracts['award_date'].isnull().sum() / contracts.shape[0])
#impute the values of award date
contracts['award_date'] = np.where(contracts['award_date'].isnull(), contracts['contract_signature_date'], contracts['award_date'])
print(contracts['award_date'].isnull().sum() / contracts.shape[0])
#impute the values of award date
contracts['award_date'] = np.where(contracts['award_date'].isnull(), contracts['contract_initial_date'], contracts['award_date'])
print(contracts['award_date'].isnull().sum() / contracts.shape[0])

0.5958929502604694
0.35734035310198586
0.0


In [54]:
contracts['tender_period'] = (contracts['award_date'] - contracts['advertisement_date']).dt.days
contracts['tender_period'].describe()

count    1.907902e+06
mean    -1.939112e+01
std      1.355341e+02
min     -4.452400e+04
25%     -2.100000e+01
50%      3.000000e+00
75%      1.700000e+01
max      3.649700e+04
Name: tender_period, dtype: float64

In [55]:
sup_limit = contracts['tender_period'].quantile(0.999)
print(sup_limit)

455.3960000006482


In [56]:
inf_limit = contracts['tender_period'].quantile(0.01)
print(inf_limit)

-380.0


In [57]:
#limit tender period to 720 days
contracts['tender_period'] = np.where(contracts['tender_period'] > sup_limit, sup_limit, contracts['tender_period'])
contracts['tender_period'] = np.where(contracts['tender_period'] < inf_limit, inf_limit, contracts['tender_period'])

In [58]:
contracts.drop(columns=['award_date', 'contract_signature_date'], inplace=True)

In [59]:
contracts.isnull().sum()

file_code                        0
contract_code                    0
submission_deadline_date    651985
advertisement_date          401006
contract_end_date              326
                             ...  
corestrength_s                   0
betweenness_edge_norm            0
bs_prop_direct                   0
neighborhood_propdirect       2204
tender_period               401006
Length: 67, dtype: int64

### Contract period

In [60]:
contracts['contract_period'] = (contracts['contract_end_date'] - contracts['contract_initial_date']).dt.days

In [61]:
contracts['contract_period'].describe()

count    2.308582e+06
mean     1.116227e+02
std      1.744841e+02
min     -6.300000e+01
25%      1.000000e+01
50%      4.200000e+01
75%      1.580000e+02
max      3.031500e+04
Name: contract_period, dtype: float64

In [62]:
#when contract period is greater than 0.999 quantile, then change it to that value
contracts['contract_period'] = np.where(contracts['contract_period'] >= contracts['contract_period'].quantile(0.999), contracts['contract_period'].quantile(0.999), contracts['contract_period'])

In [63]:
contracts.drop(columns=['contract_end_date', 'contract_initial_date'], inplace=True)

## legal limit

Should I include direct_failed_open?

In [64]:
contracts['legal_framework'].value_counts(dropna=False, normalize=True)

national         0.788564
international    0.173448
NaN              0.029391
other            0.008598
Name: legal_framework, dtype: float64

In [65]:
#submission period are days of difference between submission deadline date and advertisement date
contracts['submission_period'] = (contracts['submission_deadline_date'] - contracts['advertisement_date']).dt.days
#if sumbission period is greater than 365 days, then change it to 365 days
contracts['submission_period'] = np.where(contracts['submission_period'] > 365, 365, contracts['submission_period'])
#if sumbission period is less than -365 days, then change it to -365 days
contracts['submission_period'] = np.where(contracts['submission_period'] < -365, -365, contracts['submission_period'])

In [66]:
#procedures_column = 'procedure_type'
#procedures_to_include = ['open']

procedures_column = 'procedure_type_fixed'
procedures_to_include = ['open', 'direct_failed_open']

contracts['compliant_submission_period'] = None
conditions = [
    (contracts[procedures_column].isin(procedures_to_include) ) & (contracts['submission_period'] < 10),
    (contracts[procedures_column].isin(procedures_to_include) ) & (contracts['legal_framework'] == 'national') & (contracts['submission_period'] >= 10) & (contracts['submission_period'] < 15),
    (contracts[procedures_column].isin(procedures_to_include) ) & (contracts['legal_framework'] == 'national') & (contracts['submission_period'] >= 15),
    (contracts[procedures_column].isin(procedures_to_include) ) & (contracts['legal_framework'] == 'international') & (contracts['submission_period'] >= 10) & (contracts['submission_period'] < 15),
    (contracts[procedures_column].isin(procedures_to_include) ) & (contracts['legal_framework'] == 'international') & (contracts['submission_period'] >= 15)
]

choices = ['noncompliant', 'marginal', 'compliant', 'marginal', 'compliant']

contracts['compliant_submission_period'] = np.select(conditions, choices, default=None)

In [67]:
contracts['compliant_submission_period'].value_counts(dropna=False, normalize=True)

noncompliant    0.437266
NaN             0.423095
compliant       0.097567
marginal        0.042072
Name: compliant_submission_period, dtype: float64

In [68]:
contracts.drop(columns=['submission_deadline_date', 'advertisement_date', 'submission_period'], inplace=True)

## number of contracts

In [69]:
contracts['ncontracts_s'] = contracts.groupby(['contract_year', 'supplier_name_clean'])['supplier_name_clean'].transform('size')

In [70]:
contracts['ncontracts_b'] = contracts.groupby(['contract_year', 'purchasing_unit_id'])['supplier_name_clean'].transform('size')

# Sanctions

In [71]:
sanctioned = pd.read_feather(processed_data / 'sanctions_hypothesis.feather')

In [72]:
sanctioned

,prov_id,sanction_year_AC,sanctionedA_C_all,sanctionedA_C_max1,sanctionedA_C_max2,sanctionedA_C_max3,sanction_year_AI,sanctionedA_I_all,sanctionedA_I_max1,sanctionedA_I_max2,...,sanctionedB_C_max3,sanction_year_BI,sanctionedB_I_all,sanctionedB_I_max1,sanctionedB_I_max2,sanctionedB_I_max3,sanction_year_E_EPN,sanctionedE_EPN,sanction_year_E_AMLO,sanctionedE_AMLO
0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2308903,2308903,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2308904,2308904,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2308905,2308905,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2308906,2308906,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [73]:
sanctioned.columns

Index(['prov_id', 'sanction_year_AC', 'sanctionedA_C_all',
       'sanctionedA_C_max1', 'sanctionedA_C_max2', 'sanctionedA_C_max3',
       'sanction_year_AI', 'sanctionedA_I_all', 'sanctionedA_I_max1',
       'sanctionedA_I_max2', 'sanctionedA_I_max3', 'sanction_year_BC',
       'sanctionedB_C_all', 'sanctionedB_C_max1', 'sanctionedB_C_max2',
       'sanctionedB_C_max3', 'sanction_year_BI', 'sanctionedB_I_all',
       'sanctionedB_I_max1', 'sanctionedB_I_max2', 'sanctionedB_I_max3',
       'sanction_year_E_EPN', 'sanctionedE_EPN', 'sanction_year_E_AMLO',
       'sanctionedE_AMLO'],
      dtype='object')

In [74]:
#columns that start with 'sanction_year'
vars2drop = sanctioned.columns[sanctioned.columns.str.startswith('sanction_year')]
sanctioned = sanctioned.drop(columns=vars2drop)

In [75]:
contracts = contracts.merge(sanctioned, on = 'prov_id', how = 'left')

In [76]:
contracts.columns

Index(['file_code', 'contract_code', 'contract_price_mx', 'government_level',
       'legal_framework', 'legal_fundament_simplified', 'procedure_type_fixed',
       'procedure_venue', 'supplier_size', 'supply_type', 'prov_id', 'mad',
       'rf_submission_period', 'rf_decision_period', 'rf_procedure_type',
       'rf_bl_conformity', 'rf_buyer_dependence', 'rf_single_bidder',
       'buyer_cri', 'supplier_cri', 'bs_cri', 'neighbourhood_cri',
       'entropy_buyer_dependence', 'unweighted_competitive_clustering',
       'unbounded_weighted_competitive_clustering',
       'bounded_weighted_competitive_clustering', 'supplier_name_clean',
       'purchasing_unit_id', 'contract_year', 'prop_b_direct',
       'prop_b_failedopen', 'prop_b_direct_full', 'prop_s_direct',
       'prop_s_failedopen', 'prop_s_direct_full', 'notinRUPC',
       'ncontracts_bsw', 'active_weeks_bs', 'spending_bs_aw', 'ncontracts_bs',
       'ncontracts_bs_odds', 'fragmented_contract_odds', 'prop_contracts_bs',
       '

## Export

In [77]:
contracts.drop(columns=['prov_id', 'procedure_type_fixed'], inplace=True)

In [78]:
contracts['supplier_name_clean'].unique().shape[0]

259534

In [79]:
contracts['purchasing_unit_id'].unique().shape[0]

5036

In [80]:
contracts['contract_code'] = contracts['contract_code'].astype(str)

In [81]:
contracts['contract_year'].value_counts(dropna=False)

2017    234067
2016    231246
2015    220647
2021    209886
2019    196676
2014    195591
2018    194078
2022    193771
2013    177946
2012    167311
2020    163656
2011    124033
Name: contract_year, dtype: int64

In [82]:
#save to feather
contracts.to_feather(processed_data / 'contracts_allfeatures.feather')
